In [2]:
import pandas as pd

# -----------------------------
# Load Dataset
# -----------------------------
csv_path = "Final_Augmented_dataset_Diseases_and_Symptoms.csv"

df = pd.read_csv(csv_path)

print(f"Original Shape : {df.shape}")
print(f"Original Diseases : {df['diseases'].nunique()}")

# -----------------------------
# Target Diseases
# -----------------------------
target_diseases = [
    "atelectasis",
    "pneumothorax",
    "pneumonia",
    "pleural effusion",
    "pulmonary fibrosis",
    "emphysema",
    "hiatal hernia",
    "pulmonary congestion"
]

# -----------------------------
# Filter Dataset
# -----------------------------
filtered_df = df[df["diseases"].str.lower().isin(target_diseases)].copy()

# -----------------------------
# Results
# -----------------------------
print("\nFiltered Shape :", filtered_df.shape)
print("Remaining Diseases :", filtered_df["diseases"].nunique())

print("\nDisease Counts:")
print(filtered_df["diseases"].value_counts())

# -----------------------------
# Save
# -----------------------------
output_file = "bioBERT_8_diseases.csv"
filtered_df.to_csv(output_file, index=False)

print(f"\nFiltered dataset saved as: {output_file}")

Original Shape : (246945, 378)
Original Diseases : 773

Filtered Shape : (3766, 378)
Remaining Diseases : 8

Disease Counts:
diseases
pneumonia               1212
hiatal hernia            906
pleural effusion         607
pulmonary congestion     497
pneumothorax             296
pulmonary fibrosis       118
atelectasis               99
emphysema                 31
Name: count, dtype: int64

Filtered dataset saved as: bioBERT_8_diseases.csv


In [5]:
# now we will try removing useless symtoms taht done makes sense for these chest diseases 
# so here these symptoms never occur for any of the 8 chest diseases but we wont remove them we will keep them
# and when we will have text dataset so we will only use column that are 1 like 
# this patient has fever,cough,chest pain because fever,cough,chest pain column was 1


In [4]:
symptom_df = filtered_df.drop(columns=["diseases"])

always_zero = symptom_df.columns[(symptom_df == 0).all(axis=0)]

print(f"Always-zero columns: {len(always_zero)}")

print("\nFirst 50 always-zero symptoms:")
print(always_zero[:50].tolist())

Always-zero columns: 343

First 50 always-zero symptoms:
['anxiety and nervousness', 'depression', 'depressive or psychotic symptoms', 'insomnia', 'abnormal involuntary movements', 'palpitations', 'irregular heartbeat', 'breathing fast', 'hoarse voice', 'difficulty speaking', 'throat swelling', 'diminished hearing', 'lump in throat', 'throat feels tight', 'skin swelling', 'retention of urine', 'groin mass', 'leg pain', 'hip pain', 'suprapubic pain', 'blood in stool', 'lack of growth', 'elbow weakness', 'back weakness', 'pus in sputum', 'symptoms of the scrotum and testes', 'swelling of scrotum', 'pain in testicles', 'flatulence', 'pus draining from ear', 'jaundice', 'mass in scrotum', 'white discharge from eye', 'irritable infant', 'abusing alcohol', 'hostile behavior', 'feeling ill', 'diarrhea', 'vaginal itching', 'vaginal dryness', 'painful urination', 'involuntary urination', 'pain during intercourse', 'frequent urination', 'lower abdominal pain', 'vaginal discharge', 'blood in urin

In [6]:
# now we get this dataset into text dataset because right now the dataset is perfect for the traditional ML but bio bert expects the 
#text dataset with labels

In [7]:
import pandas as pd

# --------------------------------------------------
# Load filtered dataset
# --------------------------------------------------
INPUT_CSV = "bioBERT_8_diseases.csv"
OUTPUT_CSV = "BioBERT_Text_Dataset.csv"

df = pd.read_csv(INPUT_CSV)

print(f"Loaded dataset: {df.shape}")

# --------------------------------------------------
# Function to convert one row into natural text
# --------------------------------------------------
def row_to_text(row):
    symptoms = []

    for col in df.columns:
        if col == "diseases":
            continue

        if row[col] == 1:
            symptom = col.replace("_", " ").strip()
            symptoms.append(symptom)

    if len(symptoms) == 0:
        sentence = "No significant symptoms reported."
    elif len(symptoms) == 1:
        sentence = f"The patient presents with {symptoms[0]}."
    elif len(symptoms) == 2:
        sentence = (
            f"The patient presents with {symptoms[0]} "
            f"and {symptoms[1]}."
        )
    else:
        sentence = (
            "The patient presents with "
            + ", ".join(symptoms[:-1])
            + f", and {symptoms[-1]}."
        )

    return sentence


# --------------------------------------------------
# Generate text
# --------------------------------------------------
texts = df.apply(row_to_text, axis=1)

# --------------------------------------------------
# Create new dataframe
# --------------------------------------------------
text_df = pd.DataFrame({
    "text": texts,
    "disease": df["diseases"]
})

# --------------------------------------------------
# Save
# --------------------------------------------------
text_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved to: {OUTPUT_CSV}")

print("\nDataset Shape:")
print(text_df.shape)

print("\nFirst 10 examples:\n")

for i in range(min(10, len(text_df))):
    print("=" * 80)
    print("Disease :", text_df.iloc[i]["disease"])
    print("Text    :", text_df.iloc[i]["text"])

Loaded dataset: (3766, 378)

Saved to: BioBERT_Text_Dataset.csv

Dataset Shape:
(3766, 2)

First 10 examples:

Disease : atelectasis
Text    : The patient presents with dizziness, sore throat, cough, and headache.
Disease : atelectasis
Text    : The patient presents with shortness of breath, dizziness, and headache.
Disease : atelectasis
Text    : The patient presents with shortness of breath, sore throat, and headache.
Disease : atelectasis
Text    : The patient presents with shortness of breath, dizziness, sore throat, and headache.
Disease : atelectasis
Text    : The patient presents with dizziness, sore throat, cough, and headache.
Disease : atelectasis
Text    : The patient presents with shortness of breath, sharp chest pain, cough, and headache.
Disease : atelectasis
Text    : The patient presents with shortness of breath, sharp chest pain, dizziness, sore throat, and cough.
Disease : atelectasis
Text    : The patient presents with shortness of breath, sharp chest pain, dizziness

In [8]:
#inspect exactly which symptoms are associated with each disease on average.

In [9]:
import pandas as pd

df = pd.read_csv("bioBERT_8_diseases.csv")

symptom_cols = [c for c in df.columns if c != "diseases"]

for disease in sorted(df["diseases"].unique()):

    print("\n" + "="*70)
    print(disease.upper())

    subset = df[df["diseases"] == disease]

    freq = subset[symptom_cols].mean()

    freq = freq.sort_values(ascending=False)

    print(freq.head(20))


ATELECTASIS
headache                             0.858586
dizziness                            0.828283
sore throat                          0.707071
shortness of breath                  0.626263
cough                                0.616162
sharp chest pain                     0.595960
depression                           0.000000
depressive or psychotic symptoms     0.000000
insomnia                             0.000000
abnormal involuntary movements       0.000000
hand or finger cramps or spasms      0.000000
mass on vulva                        0.000000
jaw pain                             0.000000
itching of scrotum                   0.000000
postpartum problems of the breast    0.000000
eyelid retracted                     0.000000
hesitancy                            0.000000
elbow lump or mass                   0.000000
muscle weakness                      0.000000
throat redness                       0.000000
dtype: float64

EMPHYSEMA
shortness of breath                  0.96